# Previsão de Consumo em Sines — **XGBoost**

Gradient Boosting tabular — testa 4 conjuntos de features e compara.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4.5)

DATA_PATH = os.path.join("data", "dataset_completo_sines.xlsx")
df = pd.read_excel(DATA_PATH)

# Converter datetime corretamente (sem unit='ns' — bug do código original)
df["Data/Hora"] = pd.to_datetime(df["Data/Hora"])
df = df.sort_values("Data/Hora").set_index("Data/Hora")

# Renomear coluna do código postal de Sines
if 7520 in df.columns:
    df = df.rename(columns={7520: "Sines_Consumo"})
df["Sines_Consumo"] = pd.to_numeric(df["Sines_Consumo"], errors="coerce")

print("Período:", df.index.min(), "->", df.index.max())
print("Shape:", df.shape)
df[["Sines_Consumo"]].describe().T

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def safe_mape(y_true, y_pred, eps=1.0):
    """MAPE robusto: ignora valores próximos de zero para evitar divisões inválidas."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    mask = np.abs(y_true) > eps
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate(y_true, y_pred, nome="Modelo"):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = safe_mape(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"\n=== {nome} ===")
    print(f"MAE : {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"MAPE: {mape:.2f}%")
    print(f"R²  : {r2:.3f}")
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2}

In [ ]:
# Features tabulares
df["lag_1h"]  = df["Sines_Consumo"].shift(1)
df["lag_12h"] = df["Sines_Consumo"].shift(12)
df["lag_24h"] = df["Sines_Consumo"].shift(24)

df["ma_3h"]   = df["Sines_Consumo"].rolling(window=3).mean()
df["ma_12h"]  = df["Sines_Consumo"].rolling(window=12).mean()
df["ma_24h"]  = df["Sines_Consumo"].rolling(window=24).mean()

cyclical_features = ["Hora_sin","Hora_cos","Dia_Semana_sin","Dia_Semana_cos","Mes_sin","Mes_cos"]
categorical_features = ["Fim_de_Semana","Estacao_Inverno","Estacao_Outono","Estacao_Primavera","Estacao_Verão"]
categorical_features = [c for c in categorical_features if c in df.columns]

feature_sets = {
    "Somente_ciclicas": cyclical_features + categorical_features,
    "Lags":             ["lag_1h","lag_12h","lag_24h"] + cyclical_features + categorical_features,
    "Medias_moveis":    ["ma_3h","ma_12h","ma_24h"] + cyclical_features + categorical_features,
    "Tudo":             ["lag_1h","lag_12h","lag_24h","ma_3h","ma_12h","ma_24h"] + cyclical_features + categorical_features,
}

df_model = df.dropna(subset=["Sines_Consumo","lag_1h","lag_12h","lag_24h","ma_3h","ma_12h","ma_24h"]).copy()
print("Shape df_model:", df_model.shape)

In [ ]:
# Split cronológico 80/20
n = len(df_model)
split_idx = int(n * 0.8)
split_ts  = df_model.index[split_idx]
print("Split em:", split_ts)

y = df_model["Sines_Consumo"]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

X_train_dict, X_test_dict = {}, {}
for name, feats in feature_sets.items():
    X = df_model[feats]
    X_train_dict[name] = X.iloc[:split_idx]
    X_test_dict[name]  = X.iloc[split_idx:]

In [ ]:
from xgboost import XGBRegressor

results, y_preds, models = {}, {}, {}

for name in feature_sets:
    print(f"\n>>> Treinando XGBoost — {name}")
    m = XGBRegressor(
        n_estimators=800, max_depth=5, learning_rate=0.03,
        subsample=0.9, colsample_bytree=0.9,
        reg_alpha=0.1, reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=42, n_jobs=-1,
    )
    m.fit(X_train_dict[name], y_train)
    y_p = m.predict(X_test_dict[name])
    models[name] = m
    y_preds[name] = y_p
    results[name] = evaluate(y_test.values, y_p, nome=f"XGBoost — {name}")

In [ ]:
results_df = pd.DataFrame(results).T
print("\n=== Comparação dos modelos XGBoost ===")
print(results_df.round(3))

In [ ]:
fig, axes = plt.subplots(len(y_preds), 2, figsize=(15, 3.2*len(y_preds)))
for i, (name, y_p) in enumerate(y_preds.items()):
    ax1, ax2 = axes[i]
    ax1.plot(y_test.index, y_test.values, label="Real", linewidth=0.9)
    ax1.plot(y_test.index, y_p, label="Previsto", linewidth=0.9, alpha=0.85)
    ax1.set_title(f"XGBoost ({name}) — completo")
    ax1.set_ylabel("kWh"); ax1.legend(); ax1.grid(True, alpha=0.3)

    z = 200
    ax2.plot(y_test.index[:z], y_test.values[:z], label="Real", linewidth=1.2)
    ax2.plot(y_test.index[:z], y_p[:z], label="Previsto", linewidth=1.2)
    ax2.set_title(f"XGBoost ({name}) — zoom 200 pts")
    ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
from xgboost import plot_importance

best_name = results_df["MAE"].idxmin()
print(f"Melhor configuração (menor MAE): {best_name}")
fig, ax = plt.subplots(figsize=(10, 6))
plot_importance(models[best_name], max_num_features=15, importance_type="gain",
                grid=False, ax=ax)
ax.set_title(f"Importância das Features — XGBoost ({best_name})")
plt.tight_layout(); plt.show()

In [ ]:
os.makedirs("results", exist_ok=True)
results_df.to_csv("results/metrics_xgboost.csv")
print("Resultados guardados em results/metrics_xgboost.csv")
print(results_df.round(3))

---
### Resumo

- O XGBoost com **`Tudo`** (lags + médias móveis + cíclicas + categóricas) é tipicamente o melhor.
- Lags (`lag_1h`, `lag_24h`) e médias móveis (`ma_3h`, `ma_24h`) tendem a dominar a importância.
- Compara estes resultados com os de `02_LSTM` e `03_GRU` (ficheiros em `results/metrics_*.csv`).